# Stocks with ≥ 10,000% returns — drawdown and time

Standalone scan. It does **not** use V5 entry signals.

NSE data is stored as **one parquet per trading day** (`nse_cm_YYYYMMDD.parquet`), with every listed name in that file. A previous version scanned each file in isolation, so every symbol had **one bar** and nothing could pass the length filter. This notebook **stitches the full history per symbol** first.

**10,000% return** means close reaches **101×** entry close.

**Path:** buy at the expanding-min close; stop at the first later close ≥ 101× that low. Drawdown is only between those two dates.


In [ ]:
from pathlib import Path
import warnings, gc
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from quant.config import DATA_DIR as QUANT_DATA_DIR, PARQUET_DATA_DIR

DATA_DIR = PARQUET_DATA_DIR
OUT_DIR = QUANT_DATA_DIR / "results" / "hundred_baggers"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PANEL_PATH = OUT_DIR / "_panel_ohlc.parquet"
FEATURES_PATH = QUANT_DATA_DIR / "results" / "5x_turnaround_v5" / "features" / "stock_features.parquet"

MIN_RETURN = 100.0
TARGET_MULT = 1.0 + MIN_RETURN
MIN_BARS = 60
MIN_ENTRY_PRICE = 1.0

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 80)
pd.options.mode.copy_on_write = True

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print(f"Target: +{MIN_RETURN:.0f}%  ({TARGET_MULT:.0f}x)")
assert DATA_DIR.exists(), f"Missing parquet dir: {DATA_DIR}"


In [ ]:
def first_series(frame, name):
    s = frame[name]
    return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s

def normalize_ohlc(df, source_name):
    cols = {str(c).lower().strip(): c for c in df.columns}
    aliases = {
        "date": ["date", "datetime", "timestamp"],
        "symbol": ["symbol", "ticker", "security", "stock"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close"],
    }
    out = {}
    for target, names in aliases.items():
        found = next((cols[n] for n in names if n in cols), None)
        if found is not None:
            out[target] = found
    missing = [x for x in ["date", "high", "low", "close"] if x not in out]
    if missing:
        raise ValueError(f"{source_name}: missing {missing}; columns={list(df.columns)}")
    symbol = first_series(df, out["symbol"]) if "symbol" in out else Path(source_name).stem
    x = pd.DataFrame({
        "Date": pd.to_datetime(first_series(df, out["date"]), errors="coerce"),
        "Symbol": pd.Series(symbol, index=df.index).astype(str).str.upper().str.strip(),
        "High": pd.to_numeric(first_series(df, out["high"]), errors="coerce"),
        "Low": pd.to_numeric(first_series(df, out["low"]), errors="coerce"),
        "Close": pd.to_numeric(first_series(df, out["close"]), errors="coerce"),
    })
    x = x.dropna(subset=["Date", "Close"])
    if getattr(x["Date"].dt, "tz", None) is not None:
        x["Date"] = x["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    x = x.loc[x["Close"] > 0]
    for c in ["High", "Low", "Close"]:
        x[c] = x[c].astype("float32")
    x["Symbol"] = x["Symbol"].astype("string")
    return x

PANEL_SCHEMA = pa.schema([
    pa.field("Date", pa.timestamp("ns"), nullable=True),
    pa.field("Symbol", pa.large_string(), nullable=True),
    pa.field("High", pa.float32(), nullable=True),
    pa.field("Low", pa.float32(), nullable=True),
    pa.field("Close", pa.float32(), nullable=True),
])

def max_dd_on_path(close, high, low):
    c = np.asarray(close, dtype=np.float64)
    h = np.asarray(high, dtype=np.float64)
    lo = np.asarray(low, dtype=np.float64)
    peak_c = np.maximum.accumulate(c)
    dd_c = float(np.nanmin(c / np.maximum(peak_c, 1e-12) - 1.0))
    peak_hl = np.maximum.accumulate(np.fmax(c, np.where(np.isfinite(h), h, c)))
    trough = np.where(np.isfinite(lo), lo, c)
    rel = trough / np.maximum(peak_hl, 1e-12) - 1.0
    dd_l = float(np.nanmin(rel))
    worst_i = int(np.nanargmin(rel))
    return dd_c, dd_l, worst_i

def first_target_run(dates, close, high, low, symbol):
    c = np.asarray(close, dtype=np.float64)
    n = len(c)
    if n < MIN_BARS:
        return None, 0.0, n
    rmin = np.inf
    rmin_i = -1
    max_mult = 0.0
    hit = None
    for i in range(n):
        px = c[i]
        if not np.isfinite(px) or px <= 0:
            continue
        if px < rmin:
            rmin = px
            rmin_i = i
        if rmin_i >= 0 and rmin > 0:
            max_mult = max(max_mult, px / rmin)
        if hit is None and rmin_i >= 0 and px >= rmin * TARGET_MULT:
            sl = slice(rmin_i, i + 1)
            dd_c, dd_l, w_i = max_dd_on_path(c[sl], high[sl], low[sl])
            d0 = pd.Timestamp(dates[rmin_i])
            d1 = pd.Timestamp(dates[i])
            hit = {
                "Symbol": symbol,
                "EntryDate": d0,
                "HitDate": d1,
                "EntryClose": float(rmin),
                "HitClose": float(px),
                "Return": float(px / rmin - 1.0),
                "ReturnPct": float((px / rmin - 1.0) * 100.0),
                "Multiple": float(px / rmin),
                "TradingDays": int(i - rmin_i),
                "CalendarDays": int((d1 - d0).days),
                "Years": float((d1 - d0).days / 365.25),
                "MaxDrawdownClose": dd_c,
                "MaxDrawdownLow": dd_l,
                "WorstDDDate": pd.Timestamp(dates[rmin_i + w_i]),
                "BarsInSample": n,
                "SampleStart": pd.Timestamp(dates[0]),
                "SampleEnd": pd.Timestamp(dates[-1]),
                "MaxMultipleFromExpandingMin": float(max_mult),
                "LowEntryFlag": bool(rmin < MIN_ENTRY_PRICE),
            }
    if hit is not None:
        hit["MaxMultipleFromExpandingMin"] = float(max_mult)
    return hit, float(max_mult), n


In [ ]:
stock_files = sorted(
    p for p in DATA_DIR.rglob("*.parquet")
    if "results" not in p.parts and "indices" not in p.parts
)
print("Stock parquet files:", len(stock_files))
if stock_files:
    sample = pd.read_parquet(stock_files[0])
    print("Sample file:", stock_files[0].name, "rows:", len(sample), "cols:", list(sample.columns)[:12])
    print("Unique dates in sample file:", pd.to_datetime(sample.iloc[:, 0], errors="coerce").nunique() if len(sample) else 0)
    del sample

panel_src = None
if FEATURES_PATH.exists():
    panel_src = "features"
    print("Using existing feature panel:", FEATURES_PATH)
elif not stock_files:
    raise FileNotFoundError(f"No stock parquet under {DATA_DIR}")
else:
    if PANEL_PATH.exists():
        PANEL_PATH.unlink()
    writer = None
    n_ok = n_err = 0
    peek_dates = []
    for i, f in enumerate(stock_files, 1):
        try:
            raw = pd.read_parquet(f)
            x = normalize_ohlc(raw, f.name)
            del raw
            if x.empty:
                n_err += 1
                continue
            if len(peek_dates) < 5:
                peek_dates.append((f.name, int(x["Date"].nunique()), len(x), int(x["Symbol"].nunique())))
            table = pa.Table.from_pandas(x[["Date", "Symbol", "High", "Low", "Close"]], schema=PANEL_SCHEMA, preserve_index=False)
            del x
            if writer is None:
                writer = pq.ParquetWriter(PANEL_PATH, PANEL_SCHEMA, compression="zstd")
            writer.write_table(table)
            del table
            n_ok += 1
            if i % 250 == 0:
                gc.collect()
                print(f"stitched {i}/{len(stock_files)} files")
        except Exception as e:
            n_err += 1
            if n_err <= 8:
                print("skip", f.name, e)
    if writer is not None:
        writer.close()
    gc.collect()
    print("Stitched files OK:", n_ok, "errors:", n_err)
    print("Per-file layout (name, unique_dates, rows, symbols):", peek_dates)
    panel_src = "stitched"

if panel_src == "features":
    panel = pd.read_parquet(FEATURES_PATH, columns=["Date", "Symbol", "High", "Low", "Close"])
else:
    panel = pd.read_parquet(PANEL_PATH, columns=["Date", "Symbol", "High", "Low", "Close"])

panel["Date"] = pd.to_datetime(panel["Date"])
panel["Symbol"] = panel["Symbol"].astype("string")
print("Panel rows:", len(panel), "symbols:", panel["Symbol"].nunique(),
      "dates:", panel["Date"].min(), "→", panel["Date"].max())
print("Median bars per symbol:", float(panel.groupby("Symbol", observed=True).size().median()))


In [ ]:
rows = []
mult_rows = []
for sym, g in panel.groupby("Symbol", sort=False, observed=True):
    g = g.sort_values("Date").drop_duplicates("Date", keep="last")
    rec, max_mult, n = first_target_run(
        g["Date"].to_numpy(),
        g["Close"].to_numpy(),
        g["High"].to_numpy(),
        g["Low"].to_numpy(),
        str(sym),
    )
    mult_rows.append({"Symbol": str(sym), "Bars": n, "MaxMultipleFromExpandingMin": max_mult})
    if rec is not None:
        rows.append(rec)

del panel
gc.collect()

mults = pd.DataFrame(mult_rows)
hits = pd.DataFrame(rows)
print("Symbols scanned:", len(mults))
print("Max multiple percentiles:")
print(mults["MaxMultipleFromExpandingMin"].replace(0, np.nan).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
for m, label in [(2, "100%"), (11, "1,000%"), (51, "5,000%"), (101, "10,000%")]:
    n = int((mults["MaxMultipleFromExpandingMin"] >= m).sum())
    print(f"  names with ≥ {label} from expanding-min close ({m:.0f}x): {n}")
print("Hits at +10,000%:", len(hits))
mults.sort_values("MaxMultipleFromExpandingMin", ascending=False).head(30).to_csv(OUT_DIR / "top_multiples_all_names.csv", index=False)


## Results

Only names that actually printed a +10,000% close from the expanding-min entry. Sorted by calendar days.


In [ ]:
if hits.empty:
    print("Still no +10,000% names. Inspect top multiples CSV — if 10x/50x counts are large but 101x is 0, the data simply never reached 101x from a prior low.")
    top = mults.sort_values("MaxMultipleFromExpandingMin", ascending=False).head(40)
    display(top)
else:
    hits = hits.sort_values(["CalendarDays", "MaxDrawdownLow"]).reset_index(drop=True)
    hits.to_csv(OUT_DIR / "stocks_10000pct_from_expanding_min.csv", index=False)
    hits.to_parquet(OUT_DIR / "stocks_10000pct_from_expanding_min.parquet", index=False)
    view = hits[[
        "Symbol", "EntryDate", "HitDate", "EntryClose", "HitClose",
        "ReturnPct", "Multiple", "TradingDays", "CalendarDays", "Years",
        "MaxDrawdownClose", "MaxDrawdownLow", "WorstDDDate", "LowEntryFlag",
        "MaxMultipleFromExpandingMin",
    ]].copy()
    show = view.copy()
    show["MaxDrawdownClose"] = show["MaxDrawdownClose"].map(lambda v: f"{v:.2%}")
    show["MaxDrawdownLow"] = show["MaxDrawdownLow"].map(lambda v: f"{v:.2%}")
    show["ReturnPct"] = show["ReturnPct"].map(lambda v: f"{v:,.0f}%")
    display(show)
    print("Count:", len(hits))
    print("Median calendar days:", float(hits["CalendarDays"].median()))
    print("Median trading days:", float(hits["TradingDays"].median()))
    print("Median max DD (close):", f"{hits['MaxDrawdownClose'].median():.2%}")
    print("Median max DD (low):", f"{hits['MaxDrawdownLow'].median():.2%}")
    print("Worst max DD (low):", f"{hits['MaxDrawdownLow'].min():.2%}",
          hits.loc[hits["MaxDrawdownLow"].idxmin(), "Symbol"])
    print("Fastest calendar days:", int(hits["CalendarDays"].min()),
          hits.loc[hits["CalendarDays"].idxmin(), "Symbol"])


In [ ]:
if not hits.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist(hits["CalendarDays"], bins=min(40, max(5, len(hits))), color="steelblue", edgecolor="white")
    axes[0].set_title("Calendar days to +10,000%")
    axes[1].hist(hits["MaxDrawdownLow"] * 100, bins=min(40, max(5, len(hits))), color="indianred", edgecolor="white")
    axes[1].set_title("Max DD on the path (lows)")
    axes[2].scatter(hits["CalendarDays"], hits["MaxDrawdownLow"] * 100, alpha=0.7)
    axes[2].set_xlabel("Calendar days")
    axes[2].set_ylabel("Max DD %")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "days_and_drawdown.png", dpi=120)
    plt.show()
    plt.close(fig)
    print("CSV:", OUT_DIR / "stocks_10000pct_from_expanding_min.csv")


## How to read a row

Bought `EntryClose` on `EntryDate` (lowest close up to then). First +10,000% close is `HitDate`. `MaxDrawdownLow` / `MaxDrawdownClose` is the worst peak-to-trough **on that path**.
